# ENTSO-E 2019–2026 — Colab quickstart

This notebook loads the prepared hourly master dataset produced by the GitHub Actions downstream pipeline.


In [ ]:
!pip -q install pandas pyarrow


Upload the artifact ZIP `entsoe-master-hourly-2019-2026.zip` from GitHub Actions to Colab, then run the next cell.


In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded))


In [ ]:
import zipfile, pathlib, pandas as pd
root = pathlib.Path('/content/entsoe_master')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(root)
master_file = next(root.rglob('entsoe_master_hourly_2019_2026.parquet'))
features_file = next(root.rglob('entsoe_features_hourly_2019_2026.parquet'))
df = pd.read_parquet(master_file)
features = pd.read_parquet(features_file)
df.shape, features.shape


In [ ]:
df.info()
df.head()


In [ ]:
# Data availability by zone
availability = (df.groupby('zone')
    .agg(rows=('timestamp_utc', 'size'),
         start=('timestamp_utc', 'min'),
         end=('timestamp_utc', 'max'),
         missing_price=('day_ahead_price_eur_mwh', lambda s: s.isna().sum()),
         missing_load=('actual_load_mw', lambda s: s.isna().sum())))
availability


In [ ]:
# Example analytical slice
de = df.query("zone == 'DE_LU'").copy()
de[['timestamp_utc','day_ahead_price_eur_mwh','actual_load_mw','renewable_share_of_generation','physical_net_import_mw']].tail()
